# ============================================================
# PRUEBA TECNICA - FASTFOOD ANALYTICS
# Notebook: 02_extraccion_mysql
# Descripcion: Extrae tablas desde MySQL (Aiven Cloud)
#              y las guarda en capa Bronze del Lakehouse
# Autor: Rafael Milanes
# Fecha: 2025-03
# ============================================================


In [1]:
BRONZE_VENTAS   = "Files/bronze/mysql/ventas"
BRONZE_TICKET   = "Files/bronze/mysql/ticket"
BRONZE_PRODUCT  = "Files/bronze/mysql/product"
BRONZE_TYPE     = "Files/bronze/mysql/type"
BRONZE_TIENDAS  = "Files/bronze/mysql/tiendas"

print("Rutas cargadas correctamente")

StatementMeta(, 74d3904c-1dd2-4d14-aa1f-817ba999261c, 3, Finished, Available, Finished, False)

Rutas cargadas correctamente


In [2]:
%pip install pymysql cryptography

StatementMeta(, 74d3904c-1dd2-4d14-aa1f-817ba999261c, 8, Finished, Available, Finished, True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 2.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [3]:
import pymysql
import pandas as pd

HOST     = "mysql-3b4106d0-etraining-62c0.g.aivencloud.com"
PORT     = 10185
DATABASE = "FastFood"
USER     = "user1"
PASSWORD = "AVNS_FB2xcAz2en7pG0lHIsS"

def get_connection():
    return pymysql.connect(
        host=HOST,
        port=PORT,
        database=DATABASE,
        user=USER,
        password=PASSWORD,
        ssl={"ssl_disabled": False}
    )

# Probar conexion
try:
    conn = get_connection()
    print("Conexion exitosa a MySQL")
    conn.close()
except Exception as e:
    print(f"Error de conexion: {e}")

StatementMeta(, 74d3904c-1dd2-4d14-aa1f-817ba999261c, 10, Finished, Available, Finished, False)

Conexion exitosa a MySQL


In [6]:
from sqlalchemy import create_engine, text

engine = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}",
    connect_args={"ssl": {"ssl_disabled": False}}
)

with engine.connect() as conn:
    result = conn.execute(text("SHOW TABLES"))
    tablas = [row[0] for row in result]

print("Tablas disponibles en FastFood:")
for t in tablas:
    print(f"  - {t}")
print(f"\nTotal: {len(tablas)} tablas")

StatementMeta(, 74d3904c-1dd2-4d14-aa1f-817ba999261c, 13, Finished, Available, Finished, False)

Tablas disponibles en FastFood:
  - Product
  - Region
  - Size
  - Ticket
  - Tiendas
  - Type
  - Ventas

Total: 7 tablas


In [7]:
tablas = {
    "Product" : "Files/bronze/mysql/product",
    "Ticket"  : "Files/bronze/mysql/ticket",
    "Tiendas" : "Files/bronze/mysql/tiendas",
    "Type"    : "Files/bronze/mysql/type",
    "Ventas"  : "Files/bronze/mysql/ventas",
    "Region"  : "Files/bronze/mysql/region",
    "Size"    : "Files/bronze/mysql/size",
}

for tabla, path in tablas.items():
    try:
        df = pd.read_sql(f"SELECT * FROM {tabla}", engine)
        spark_df = spark.createDataFrame(df)
        spark_df.write.format("delta").mode("overwrite").save(path)
        print(f"OK {tabla}: {df.shape[0]} filas, {df.shape[1]} columnas")
    except Exception as e:
        print(f"ERROR en {tabla}: {e}")

StatementMeta(, 74d3904c-1dd2-4d14-aa1f-817ba999261c, 14, Finished, Available, Finished, False)

OK Product: 20 filas, 2 columnas
OK Ticket: 208863 filas, 5 columnas
OK Tiendas: 10 filas, 7 columnas
OK Type: 2 filas, 2 columnas
OK Ventas: 85936 filas, 3 columnas
OK Region: 4 filas, 2 columnas
OK Size: 3 filas, 2 columnas


In [8]:
print("VERIFICACION BRONZE MYSQL\n")
for tabla, path in tablas.items():
    try:
        df_check = spark.read.format("delta").load(path)
        print(f"OK {tabla}: {df_check.count()} filas en Bronze")
    except Exception as e:
        print(f"ERROR {tabla}: {e}")

StatementMeta(, 74d3904c-1dd2-4d14-aa1f-817ba999261c, 15, Finished, Available, Finished, False)

VERIFICACION BRONZE MYSQL

OK Product: 20 filas en Bronze
OK Ticket: 208863 filas en Bronze
OK Tiendas: 10 filas en Bronze
OK Type: 2 filas en Bronze
OK Ventas: 85936 filas en Bronze
OK Region: 4 filas en Bronze
OK Size: 3 filas en Bronze
